[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/antfolk/BMEN35_2026/blob/main/Session3/BMEN35_Ex3_02_CNN_MNIST_step_by_step.ipynb)

# Convolutional neural networks: compare two models on MNIST

In this exercise we will train a small convolutional neural network (CNN) to recognize handwritten digits. First, let's inspect the image array and labels. Then we will add the model layers one by one, train the network, and look at its errors.

MNIST is a small grayscale image dataset. We use a subset so the example is practical in Colab. The aim is to understand what the layers do, not to achieve a record score.

A GPU is optional for this subset. Run the cells in order and look at the image examples before training.

## 1. Import TensorFlow/Keras and plotting tools

Colab includes TensorFlow. If you use a local environment, installing a compatible TensorFlow version is your responsibility. We will train two small CNNs with the same data and training settings. The only architecture difference is that Model B has one extra convolution layer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay
)

print('TensorFlow version:', tf.__version__)
print('GPU visible to TensorFlow:', bool(tf.config.list_physical_devices('GPU')))

## 2. Load and inspect MNIST

Keras returns training and test images separately. One image is a 28-by-28 array of pixel intensities. Its label is an integer from 0 to 9. Before changing the data, inspect its shape and display a few examples.

In [ ]:
(all_train_images, all_train_labels), (all_test_images, all_test_labels) = keras.datasets.mnist.load_data()

print('Original training image array:', all_train_images.shape)
print('Original training labels:', all_train_labels.shape)
print('Original test image array:', all_test_images.shape)
print('Minimum and maximum pixel values:', all_train_images.min(), all_train_images.max())

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
# ravel() lets the loop visit each subplot in a simple one-dimensional sequence.
for ax, image, label in zip(axes.ravel(), all_train_images[:10], all_train_labels[:10]):
    ax.imshow(image, cmap='gray_r', vmin=0, vmax=255)
    ax.set_title(f'label: {label}')
    ax.axis('off')
plt.suptitle('A few MNIST training images')
plt.tight_layout()
plt.show()

## 3. Choose training, validation, and test subsets

We use the first 10,000 training images and first 2,000 test images so the example runs more quickly. We keep 1,000 of the selected training images for validation. Both CNNs will use exactly the same training, validation, and test observations.

In [ ]:
number_of_selected_training_images = 10000
number_of_validation_images = 1000
number_of_test_images = 2000

# First select a fixed subset from the MNIST training images.
x_selected = all_train_images[:number_of_selected_training_images]
y_selected = all_train_labels[:number_of_selected_training_images]

# Keep the last 1,000 selected examples aside for validation.
x_train = x_selected[:-number_of_validation_images]
y_train = y_selected[:-number_of_validation_images]
x_validation = x_selected[-number_of_validation_images:]
y_validation = y_selected[-number_of_validation_images:]

# Select a separate test subset, which we save for the final check.
x_test = all_test_images[:number_of_test_images]
y_test = all_test_labels[:number_of_test_images]

print('Training images:', x_train.shape, '| labels:', y_train.shape)
print('Validation images:', x_validation.shape, '| labels:', y_validation.shape)
print('Test images:', x_test.shape, '| labels:', y_test.shape)
print('Training class counts:', np.bincount(y_train, minlength=10))

## 4. Prepare the image tensors

Before this step, one image has shape (28, 28). A convolution layer also needs a channel dimension. MNIST images are grayscale, so each image has one channel. Keras uses (batch, height, width, channels) by default.

We divide pixel values by 255 to convert them to the range 0–1, then add a final channel dimension. The pixels themselves do not change when we add that dimension.

In [ ]:
# Convert integer pixel values from 0–255 to floating-point values from 0–1.
x_train = x_train.astype('float32') / 255.0
x_validation = x_validation.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Add one grayscale channel: (N, 28, 28) becomes (N, 28, 28, 1).
x_train = x_train[..., np.newaxis]
x_validation = x_validation[..., np.newaxis]
x_test = x_test[..., np.newaxis]

print('Prepared training tensor:', x_train.shape)
print('Prepared validation tensor:', x_validation.shape)
print('Prepared test tensor:', x_test.shape)
print('Prepared pixel range:', x_train.min(), 'to', x_train.max())

## 5. Build Model A: one convolution layer

Model A starts with one convolution layer with 16 filters. A 2×2 max-pooling layer then reduces the feature maps from 28×28 to 14×14. We will add the layers one at a time and inspect the output shape after each step.

In [ ]:
model_a = keras.Sequential([
    # One MNIST image has height 28, width 28, and one grayscale channel.
    layers.Input(shape=(28, 28, 1))
])
model_a.summary()

### Add the first convolution layer

In [ ]:
model_a.add(layers.Conv2D(
    filters=16,
    kernel_size=3,
    activation='relu',
    padding='same'
))
model_a.summary()

### Reduce the feature-map size

In [ ]:
model_a.add(layers.MaxPooling2D(pool_size=2))
model_a.summary()

### Flatten the feature maps, then add the classifier

In [ ]:
model_a.add(layers.Flatten())
model_a.summary()

In [ ]:
model_a.add(layers.Dense(32, activation='relu'))
model_a.summary()

In [ ]:
model_a.add(layers.Dense(10, activation='softmax'))
model_a.summary()

## 6. Compile and train Model A

The labels are integer class numbers, so we use sparse categorical cross-entropy. We use the same optimizer, batch size, and number of epochs for Model B so that the comparison focuses on the architecture change.

In [ ]:
model_a.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
print('Model A is compiled and ready for training.')

### Train Model A

The validation data are evaluated after each epoch, but they are not used to update the weights directly.

In [ ]:
number_of_epochs = 4
batch_size = 128
history_a = model_a.fit(
    x_train,
    y_train,
    validation_data=(x_validation, y_validation),
    epochs=number_of_epochs,
    batch_size=batch_size,
    verbose=1
)

## 7. Build Model B: add one convolution layer

Model B keeps Model A's first convolution layer, pooling layer, and dense classifier. Its one change is an additional convolution layer with 32 filters before pooling. Compare the model summaries: the extra layer adds learned filters and increases the number of trainable parameters.

In [ ]:
model_b = keras.Sequential([
    layers.Input(shape=(28, 28, 1))
])
model_b.summary()

### Add the same first convolution layer as Model A

In [ ]:
model_b.add(layers.Conv2D(
    filters=16,
    kernel_size=3,
    activation='relu',
    padding='same'
))
model_b.summary()

### Add one new convolution layer

This is the single architecture change compared with Model A. It creates 32 feature maps while keeping the spatial size at 28×28.

In [ ]:
model_b.add(layers.Conv2D(
    filters=32,
    kernel_size=3,
    activation='relu',
    padding='same'
))
model_b.summary()

### Use the same pooling and classifier layers as Model A

In [ ]:
model_b.add(layers.MaxPooling2D(pool_size=2))
model_b.summary()

In [ ]:
model_b.add(layers.Flatten())
model_b.summary()

In [ ]:
model_b.add(layers.Dense(32, activation='relu'))
model_b.summary()

In [ ]:
model_b.add(layers.Dense(10, activation='softmax'))
model_b.summary()

## 8. Compile and train Model B

Use the same loss, optimizer, batch size, and number of epochs as Model A. That makes the validation comparison easier to interpret.

In [ ]:
model_b.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
print('Model B is compiled and ready for training.')

### Train Model B

In [ ]:
history_b = model_b.fit(
    x_train,
    y_train,
    validation_data=(x_validation, y_validation),
    epochs=number_of_epochs,
    batch_size=batch_size,
    verbose=1
)

## 9. Compare the learning curves and validation results

Training curves show how each model fits the training examples and how its performance changes on the same validation examples. We also compare validation accuracy and macro F1. MNIST is fairly balanced, but the class-wise report and confusion matrix show which digits each model still confuses.

In [ ]:
epochs = np.arange(1, number_of_epochs + 1)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0, 0].plot(epochs, history_a.history['loss'], marker='o', label='training')
axes[0, 0].plot(epochs, history_a.history['val_loss'], marker='o', label='validation')
axes[0, 0].set_title('Model A loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Sparse categorical cross-entropy')
axes[0, 0].legend()

axes[0, 1].plot(epochs, history_b.history['loss'], marker='o', label='training')
axes[0, 1].plot(epochs, history_b.history['val_loss'], marker='o', label='validation')
axes[0, 1].set_title('Model B loss')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Sparse categorical cross-entropy')
axes[0, 1].legend()

axes[1, 0].plot(epochs, history_a.history['val_accuracy'], marker='o', label='Model A')
axes[1, 0].plot(epochs, history_b.history['val_accuracy'], marker='o', label='Model B')
axes[1, 0].set_title('Validation accuracy')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Accuracy')
axes[1, 0].legend()

axes[1, 1].plot(epochs, history_a.history['accuracy'], marker='o', label='Model A')
axes[1, 1].plot(epochs, history_b.history['accuracy'], marker='o', label='Model B')
axes[1, 1].set_title('Training accuracy')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Accuracy')
axes[1, 1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Predict the validation labels with Model A.
validation_probabilities_a = model_a.predict(x_validation, verbose=0)
validation_predictions_a = np.argmax(validation_probabilities_a, axis=1)
validation_accuracy_a = accuracy_score(y_validation, validation_predictions_a)
validation_macro_f1_a = f1_score(y_validation, validation_predictions_a, average='macro')

# Predict the same validation labels with Model B.
validation_probabilities_b = model_b.predict(x_validation, verbose=0)
validation_predictions_b = np.argmax(validation_probabilities_b, axis=1)
validation_accuracy_b = accuracy_score(y_validation, validation_predictions_b)
validation_macro_f1_b = f1_score(y_validation, validation_predictions_b, average='macro')

print('Model A validation accuracy:', round(validation_accuracy_a, 3))
print('Model A validation macro F1:', round(validation_macro_f1_a, 3))
print('Model B validation accuracy:', round(validation_accuracy_b, 3))
print('Model B validation macro F1:', round(validation_macro_f1_b, 3))
print('\nModel A validation report:')
print(classification_report(y_validation, validation_predictions_a, zero_division=0))
print('Model B validation report:')
print(classification_report(y_validation, validation_predictions_b, zero_division=0))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay.from_predictions(
    y_validation, validation_predictions_a,
    labels=np.arange(10), ax=axes[0], cmap='Blues', colorbar=False
)
axes[0].set_title('Model A validation confusion matrix')
ConfusionMatrixDisplay.from_predictions(
    y_validation, validation_predictions_b,
    labels=np.arange(10), ax=axes[1], cmap='Blues', colorbar=False
)
axes[1].set_title('Model B validation confusion matrix')
plt.tight_layout()
plt.show()

## 10. Select a model, then evaluate the test subset once

Use validation macro F1 to select between the two architectures. Then evaluate the selected model on the separate test subset. Do not use the test results to choose between the models.

In [ ]:
# Choose the model with the higher validation macro F1.
if validation_macro_f1_b > validation_macro_f1_a:
    best_model = model_b
    best_model_name = 'Model B: two convolution layers'
else:
    best_model = model_a
    best_model_name = 'Model A: one convolution layer'
print('Selected using validation macro F1:', best_model_name)

# Evaluate only the selected model on the test subset.
test_probabilities = best_model.predict(x_test, verbose=0)
test_predictions = np.argmax(test_probabilities, axis=1)
print('Test accuracy:', round(accuracy_score(y_test, test_predictions), 3))
print('Test macro F1:', round(f1_score(y_test, test_predictions, average='macro'), 3))
print(classification_report(y_test, test_predictions, labels=np.arange(10), zero_division=0))
ConfusionMatrixDisplay.from_predictions(
    y_test, test_predictions, labels=np.arange(10), cmap='Blues'
)
plt.title(best_model_name + ' test confusion matrix')
plt.show()

## 11. Look at some mistakes

The confusion matrix shows which digit pairs are confused overall. The examples below show individual errors. We use test examples here only after choosing the model with validation data.

In [ ]:
# Find the positions where the selected model predicted the wrong digit.
wrong_positions = np.flatnonzero(test_predictions != y_test)
number_to_show = min(10, len(wrong_positions))

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
# ravel() turns the grid of subplot axes into a simple sequence for the loop.
for ax, position in zip(axes.ravel(), wrong_positions[:number_to_show]):
    ax.imshow(x_test[position, :, :, 0], cmap='gray_r', vmin=0, vmax=1)
    ax.set_title(f'true {y_test[position]}, predicted {test_predictions[position]}')
    ax.axis('off')
plt.suptitle('Examples the selected CNN classified incorrectly')
plt.tight_layout()
plt.show()

## 12. Specific conclusions

1. What extra calculation does Model B perform compared with Model A?
2. How did the additional convolution layer change the model summary and parameter count?
3. Which model had higher validation macro F1? Was its validation accuracy also higher?
4. What kinds of digit pairs appear most often away from the diagonal in the validation confusion matrices?
5. Did the validation results lead you to choose a different model than you expected?
6. Why did we use validation data to select a model and leave the test subset for the final evaluation?

The next exercise applies image classification to a small biomedical dataset, where class imbalance makes per-class metrics especially important.